In [ ]:
!pip install pennylane pennylane-lightning -q

# ⚛️ Kuantum Kapıları

**Dr. Buket Toptaş** | Kuantum Makine Öğrenmesi Dersi

**Konular:** Pauli Kapıları (X, Y, Z) · Hadamard · Rotasyon Kapıları (Rx, Ry, Rz) · CNOT · Çok-Qubit Kapıları

---
## 1. Kuantum Kapısı Nedir?

Qubit'e uygulanan **dönüşüm**. Klasik AND/OR/NOT'un kuantum karşılığı.

Fark: Kuantum kapıları her zaman **terslenebilir** (unitary). Bilgi kaybolmaz.

---
## 2. Pauli Kapıları (X, Y, Z)

| Kapı | Matris | Etki |
|------|--------|------|
| X (NOT) | [[0,1],[1,0]] | \|0⟩↔\|1⟩ çevirir |
| Y | [[0,-i],[i,0]] | X+Z birleşimi |
| Z (Faz) | [[1,0],[0,-1]] | \|1⟩'in fazını çevirir |

In [ ]:
import numpy as np

X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])

ket0 = np.array([[1],[0]])
ket1 = np.array([[0],[1]])
plus = (ket0+ket1)/np.sqrt(2)

print("=== Pauli Kapılarının Etkileri ===\n")
for gname, gate in [('X',X), ('Y',Y), ('Z',Z)]:
    print(f"--- {gname} kapısı ---")
    for iname, inp in [('|0⟩',ket0), ('|1⟩',ket1), ('|+⟩',plus)]:
        out = gate @ inp
        a, b = out[0,0], out[1,0]
        print(f"  {gname}{iname} = ({a:+.2f})|0⟩ + ({b:+.2f})|1⟩")
    print()

In [ ]:
import pennylane as qml
import matplotlib.pyplot as plt

dev = qml.device('default.qubit', wires=1, shots=1000)

# Her Pauli kapısını |0⟩'a uygula ve ölç
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
gates_list = [('Yok (|0⟩)', None), ('X', qml.PauliX), ('Z', qml.PauliZ), ('H', qml.Hadamard)]

for ax, (name, gate_fn) in zip(axes, gates_list):
    @qml.qnode(dev)
    def circuit():
        if gate_fn is not None:
            gate_fn(wires=0)
        return qml.counts()
    
    res = circuit()
    vals = [res.get('0',0), res.get('1',0)]
    ax.bar(['|0⟩','|1⟩'], vals, color=['#3b82f6','#ef4444'], alpha=0.85, width=0.5, edgecolor='white')
    ax.set_title(f'{name}|0⟩', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1100)
    for j, v in enumerate(vals):
        if v > 0: ax.text(j, v+20, str(v), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Kapıların |0⟩ Üzerine Etkisi (1000 ölçüm)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Hadamard Kapısı

Süperpozisyon oluşturur. Kendi tersi: **HH = I** (iki kez uygula → başa dön).

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \quad H|0\rangle = |+\rangle, \quad H|1\rangle = |-\rangle$$

In [ ]:
import pennylane as qml

dev = qml.device('default.qubit', wires=1, shots=1000)

@qml.qnode(dev)
def h_once():
    qml.Hadamard(wires=0)
    return qml.counts()

@qml.qnode(dev)
def h_twice():
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=0)
    return qml.counts()

print("=== H|0⟩ ===")
print(h_once())
print("→ ~%50 / %50 (süperpozisyon)")

print()
print("=== HH|0⟩ ===")
print(h_twice())
print("→ %100 |0⟩ (başa döndü!)")

---
## 4. Rotasyon Kapıları (Rx, Ry, Rz)

Bloch küresinde belirli açıyla döndürme. **QML'nin temel yapı taşları!**

Variasyonel devrelerde θ parametresi eğitimle optimize edilir — sinir ağındaki ağırlıklar gibi.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

dev = qml.device('default.qubit', wires=1)

angles = np.linspace(0, 2*np.pi, 50)
prob_0 = []

for theta in angles:
    @qml.qnode(dev)
    def ry_test(t):
        qml.RY(t, wires=0)
        return qml.probs(wires=0)
    prob_0.append(ry_test(theta)[0])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(np.degrees(angles), prob_0, color='#3b82f6', lw=2.5, label='P(|0⟩)')
ax.plot(np.degrees(angles), [1-p for p in prob_0], color='#ef4444', lw=2.5, label='P(|1⟩)')

ax.axvline(x=90, color='#10b981', ls='--', alpha=0.7, label='θ=90° → süperpozisyon')
ax.axvline(x=180, color='#f59e0b', ls='--', alpha=0.7, label='θ=180° → tam çevirme')

ax.set_xlabel('θ (derece)', fontsize=12)
ax.set_ylabel('Olasılık', fontsize=12)
ax.set_title('Ry(θ) — Açıya Göre Olasılık', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("θ=0°   → |0⟩")
print("θ=90°  → (|0⟩+|1⟩)/√2")
print("θ=180° → |1⟩")
print("θ=360° → |0⟩")

---
## 5. Çok-Qubit Kapıları

### CNOT (Controlled-NOT)

En önemli 2-qubit kapısı. Dolanıklık oluşturur.

- Kontrol=0 → hedefi değiştirme
- Kontrol=1 → hedefi çevir (X uygula)

| Giriş | Çıkış | |
|:---:|:---:|---|
| \|00⟩ | \|00⟩ | |
| \|01⟩ | \|01⟩ | |
| \|10⟩ | \|11⟩ | ← çevrildi |
| \|11⟩ | \|10⟩ | ← çevrildi |

In [ ]:
import pennylane as qml

dev = qml.device('default.qubit', wires=2)

print("=== CNOT Doğruluk Tablosu ===\n")
for a in [0,1]:
    for b in [0,1]:
        @qml.qnode(dev)
        def cnot_test():
            if a: qml.PauliX(wires=0)
            if b: qml.PauliX(wires=1)
            qml.CNOT(wires=[0,1])
            return qml.probs(wires=[0,1])
        
        probs = cnot_test()
        result = int(np.argmax(probs))
        out_a, out_b = result//2, result%2
        mark = " ← çevrildi!" if a==1 and out_b!=b else ""
        print(f"  |{a}{b}⟩ → |{out_a}{out_b}⟩{mark}")

In [ ]:
# Bell durumu: H + CNOT
dev2 = qml.device('default.qubit', wires=2, shots=1000)

@qml.qnode(dev2)
def bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0,1])
    return qml.counts()

print("=== H + CNOT = Bell Durumu ===\n")
print(qml.draw(bell)())
print()
print(dict(sorted(bell().items())))
print("\n→ Sadece 00 ve 11 → DOLANIKLIK!")

---
## 6. Kapı Özet Tablosu

| Kapı | Qubit | Etki | QML'de Kullanım |
|------|:-----:|------|-----------------|
| X | 1 | Bit çevirme | Durum hazırlama |
| Z | 1 | Faz çevirme | Oracle (Grover) |
| H | 1 | Süperpozisyon | Başlangıç |
| **Ry(θ)** | 1 | θ döndürme | **Variasyonel parametre** |
| **CNOT** | 2 | Koşullu çevirme | **Dolanıklık katmanı** |

> QML'de en çok: **Ry(θ)** (parametreli) + **CNOT** (dolanıklık)